# Colorado 2020 Interactive Fire Map

Visualizes three layers for Colorado using Google Earth Engine + geemap:

| Layer | Source | Description |
|-------|--------|-------------|
| Fire Perimeters | MTBS | 2020 fire polygon boundaries |
| Fire Severity | MTBS annual severity mosaic | dNBR severity classes (0–6) |
| Vegetation / Land Cover | NLCD 2021 | 20-class land cover |

All data stays server-side — no rasters are downloaded locally.

## Step 0 — Setup

In [1]:
import sys
!{sys.executable} -m pip install earthengine-api geemap

  Obtaining dependency information for earthengine-api from https://files.pythonhosted.org/packages/b1/c0/e8d2e1f3d090f48d23ca8cc67aa90594afa9d365e26f4c482ce80a31161e/earthengine_api-1.7.26-py3-none-any.whl.metadata
  Obtaining dependency information for geemap from https://files.pythonhosted.org/packages/96/4f/fb3392a41254afc656c6dd1ea4b9936a012e451fbe5e934fbda81878a6c3/geemap-0.37.2-py3-none-any.whl.metadata
  Obtaining dependency information for google-cloud-storage from https://files.pythonhosted.org/packages/ad/ff/ca9ab2417fa913d75aae38bf40bf856bb2749a604b2e0f701b37cfcd23cc/google_cloud_storage-3.10.1-py3-none-any.whl.metadata
  Obtaining dependency information for google-api-python-client>=1.12.1 from https://files.pythonhosted.org/packages/99/c7/1817b4edf966d5afcac1c0781ca36d621bc0cb58104c4e7c2a475ab185f7/google_api_python_client-2.196.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-auth>=1.4.1 from https://files.pythonhosted.org/packages/ee/fc/2cdc7425

In [2]:
import sys
import os
sys.path.insert(0, os.getcwd())

import ee
import geemap

# Authenticate once if needed — comment out after first run
# ee.Authenticate()
ee.Initialize(project='fire-344-467415')

from fetch_interactive_map import (
    load_all_layers,
    NLCD_CLASSES, NLCD_PALETTE,
    SEVERITY_CLASSES, SEVERITY_PALETTE,
)

print('EE initialized. Loading layers...')
layers = load_all_layers(fire_year=2020)
print('Layers ready.')

EE initialized. Loading layers...
Layers ready.


## Step 1 — Build the Interactive Map

Adds all three layers with toggle controls and legends.

In [3]:
Map = geemap.Map(center=[39.0, -105.5], zoom=7)

# Layer 1: NLCD vegetation / land cover (bottom)
Map.addLayer(
    layers['nlcd'],
    layers['nlcd_vis'],
    'Vegetation / Land Cover (NLCD 2021)',
    shown=True,
    opacity=0.7,
)

# Layer 2: Fire severity raster
Map.addLayer(
    layers['severity'],
    layers['severity_vis'],
    'Fire Severity 2020 (MTBS dNBR)',
    shown=True,
    opacity=0.85,
)

# Layer 3: Fire perimeters (vector on top)
Map.addLayer(
    layers['perimeters_styled'],
    {},
    'Fire Perimeters 2020 (MTBS)',
    shown=True,
)

Map.addLayerControl()
Map

Map(center=[39.0, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Step 2 — Add Legends

In [4]:
# Fire severity legend
severity_legend = {
    label: color
    for (_, label), color in zip(SEVERITY_CLASSES.items(), SEVERITY_PALETTE)
}
Map.add_legend(
    title='Fire Severity (MTBS)',
    legend_dict=severity_legend,
    position='bottomleft',
)

# Condensed NLCD land cover legend
key_nlcd = {
    'Open Water':        '#466b9f',
    'Developed':         '#eb0000',
    'Barren':            '#b3ac9f',
    'Deciduous Forest':  '#68ab5f',
    'Evergreen Forest':  '#1c5f2c',
    'Mixed Forest':      '#b5c58f',
    'Shrub/Scrub':       '#ccb879',
    'Grassland':         '#dfdfc2',
    'Pasture/Hay':       '#dcd939',
    'Cultivated Crops':  '#ab6c28',
    'Wetlands':          '#b8d9eb',
}
Map.add_legend(
    title='Land Cover (NLCD 2021)',
    legend_dict=key_nlcd,
    position='bottomright',
)

Map

Map(bottom=12823.0, center=[39.0, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

## Step 3 — Fire Summary Table

Show each 2020 fire: name, ignition date, and area sorted by size.

In [5]:
import pandas as pd

info = layers['perimeters'].select(
    ['Incid_Name', 'Ig_Date', 'BurnBndAc', 'area_ha']
).getInfo()

rows = [f['properties'] for f in info['features']]
df_fires = pd.DataFrame(rows)
df_fires['area_ha'] = df_fires['area_ha'].round(1)
df_fires['BurnBndAc'] = df_fires['BurnBndAc'].round(1)
df_fires = df_fires.sort_values('area_ha', ascending=False).reset_index(drop=True)
df_fires.columns = ['Fire Name', 'Ignition Date', 'Area (acres)', 'Area (ha)']

print(f'Total 2020 fires in Colorado: {len(df_fires)}')
print(f'Total burned area: {df_fires["Area (ha)"].sum():,.0f} ha')
df_fires

Total 2020 fires in Colorado: 21
Total burned area: 334,839 ha


,Fire Name,Ignition Date,Area (acres),Area (ha)
0,208910,1597302000000,CAMERON PEAK,84439.4
1,188924,1602658800000,EAST TROUBLESOME,76371.2
2,179894,1600326000000,MULLEN,72703.6
3,138141,1596178800000,PINE GULCH,55849.6
4,30705,1597042800000,GRIZZLY CREEK,12412.9
5,20886,1599375600000,MIDDLE FORK,8441.5
6,14679,1597388400000,WILLIAMS FORK,5935.2
7,11823,1589958000000,CHERRY CANYON,4782.4
8,9836,1602918000000,CALWOOD,3976.4
9,3414,1591513200000,DEER CANYON,1381.1


## Step 4 — Export Map as HTML

Save the interactive map as a standalone HTML file that can be shared without Jupyter.

In [5]:
Map.save('colorado_fire_map_2020.html')
print('Map saved: colorado_fire_map_2020.html')

Map saved: colorado_fire_map_2020.html
